        # 📁 L10　檔案與例外處理
        **Python 冒險之旅 2026**　｜　Day 5（09/04 五）🏜️ 檔案之島　｜　關卡　｜　🏅 100 XP

        📖 對應教科書：第 9 章 9.1–9.5


        ### 🎯 這一關你會學到
        - 用 open() / with 讀寫文字檔
- os.path 檢查路徑、建立資料夾
- try / except / finally 處理執行時期錯誤

        ### 🧭 闖關方式
        1. 先按下方「🧰 魔法工具箱」那一格左邊的 ▶（第一次執行 Colab 會花幾秒鐘連線）。
        2. 依序閱讀說明、執行範例、完成每個「🎯 任務」，再執行它下面的「檢查」格。
        3. 看到 ✅ 就往下一個任務；看到 ❌ 就依提示修改，再重新執行任務格與檢查格。
        4. 全部通過後，執行最下面的「🔑 通關密語」格，把密語貼回 [入口網頁](https://johnnychao.github.io/python-quest-2026/)。

        > 💾 建議先點選「檔案 → 在雲端硬碟中儲存副本」，你的進度才會留在自己的 Google 雲端硬碟。

In [ ]:
#@title 🧰 魔法工具箱：先在右邊填「暱稱」，再按左邊的 ▶ 執行這一格 { display-mode: "form" }
暱稱 = "" #@param {type:"string"}
# ======================================================================
#  Python 冒險之旅 2026 · 關卡檢查工具（看不懂沒關係，這一格不是今天的功課 😉）
# ======================================================================
import hashlib, unicodedata, io, sys, re, contextlib, traceback, builtins

_LEVEL = "L10"
_SALT = "python-quest-2026-datama"
_TASKS = ["10-1", "10-2", "10-3", "10-4", "10-5"]
_XP_EACH = 20
_CHECKS = {}
_PASSED = builtins.__dict__.setdefault("_pyquest_" + _LEVEL, {})
_HINTS = {}

def _norm_name(s):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s))).lower()

def _squash(s):
    return re.sub(r"\s+", "", str(s))

def 出現(out, *subs):
    """輸出中是否（忽略空白）包含所有片段"""
    o = _squash(out)
    return all(_squash(x) in o for x in subs)

def 數字們(out):
    """抓出輸出裡所有的數字（float）"""
    return [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", str(out))]

def 行列表(out):
    return [ln.rstrip() for ln in str(out).splitlines() if ln.strip()]

class _NeedMoreInput(Exception):
    pass

_HIST = builtins.__dict__.setdefault("_pyquest_hist", [])
def _on_pre_run(*args):
    try:
        info = args[0]
        src = getattr(info, "raw_cell", None)
        if isinstance(src, str):
            _HIST.append(src)
    except Exception:
        pass
try:
    _ip = get_ipython()
    if not builtins.__dict__.get("_pyquest_hooked"):
        _ip.events.register("pre_run_cell", _on_pre_run)
        builtins.__dict__["_pyquest_hooked"] = True
except Exception:
    pass

def _history():
    try:
        ip = get_ipython()
        h = list(ip.user_ns.get("In") or ip.user_ns.get("_ih") or [])
    except Exception:
        h = list(globals().get("In") or [])
    return [c for c in (h + list(_HIST)) if isinstance(c, str)]

def _find_cell(tid):
    marker = "# 🎯 任務 " + tid
    for cell in reversed(_history()):
        if marker in cell:
            lines = [ln for ln in cell.splitlines()
                     if not re.match(r"\s*(檢查|通關密語)\s*\(", ln)]
            return "\n".join(lines)
    return None

def _make_runner(src):
    def run(*inputs):
        feed = iter([str(x) for x in inputs])
        buf = io.StringIO()
        try:
            ns = dict(get_ipython().user_ns)
        except Exception:
            ns = dict(globals())
        def _fake_input(prompt=""):
            try:
                return next(feed)
            except StopIteration:
                raise _NeedMoreInput()
        ns["input"] = _fake_input
        ns["__name__"] = "__main__"
        try:
            import matplotlib.pyplot as _plt
            _plt.close("all"); _orig_show = _plt.show; _plt.show = lambda *a, **k: None
        except Exception:
            _plt = None
        try:
            with contextlib.redirect_stdout(buf):
                exec(compile(src, "<任務 " + _LEVEL + ">", "exec"), ns)
        finally:
            if _plt is not None:
                _plt.show = _orig_show
        return buf.getvalue(), ns
    run.src = src
    return run

def 任務定義(tid, fn, 提示=""):
    _CHECKS[tid] = fn
    _HINTS[tid] = 提示

def _progress():
    done = sum(1 for t in _TASKS if _PASSED.get(t))
    bar = "■" * done + "□" * (len(_TASKS) - done)
    return f"[{bar}] {done}/{len(_TASKS)}"

def 檢查(tid):
    tid = str(tid)
    if tid not in _CHECKS:
        print(f"⚠️ 找不到任務 {tid} 的檢查設定。"); return
    src = _find_cell(tid)
    if src is None:
        print(f"❌ 找不到「# 🎯 任務 {tid}」的程式格。請先執行那一格（並保留第一行的標記），再執行這裡。")
        return
    run = _make_runner(src)
    try:
        result = _CHECKS[tid](run)
    except _NeedMoreInput:
        result = (False, "你的程式呼叫 input() 的次數比題目預期的多，請檢查輸入的次數。")
    except Exception as e:
        tb = traceback.format_exc().strip().splitlines()[-1]
        result = (False, f"程式執行時發生錯誤 → {tb}")
    ok, extra = (result, "") if isinstance(result, bool) else result
    if ok:
        first = not _PASSED.get(tid)
        _PASSED[tid] = True
        print(f"✅ 任務 {tid} 通過！{'+' + str(_XP_EACH) + ' XP ' if first else ''}{_progress()}")
        if all(_PASSED.get(t) for t in _TASKS):
            print("🏆 本關所有任務都完成了！請執行最下面的「通關密語」那一格。")
    else:
        print(f"❌ 任務 {tid} 還沒通過。{_progress()}")
        if extra: print("   💬 " + str(extra))
        if _HINTS.get(tid): print("   💡 提示：" + _HINTS[tid])
        print("   👉 修改程式後，先重新執行任務那一格，再執行這一格。")

def 通關密語():
    missing = [t for t in _TASKS if not _PASSED.get(t)]
    if missing:
        print("🔒 還有任務未通過：" + "、".join(missing) + "　完成後再來拿密語吧！")
        return
    name = 暱稱.strip() if isinstance(暱稱, str) else ""
    if not name:
        name = input("請輸入你在入口網頁登錄的暱稱：").strip()
    if not name:
        print("⚠️ 暱稱不能是空白。"); return
    code = hashlib.sha256(f"{_SALT}|{_LEVEL}|{_norm_name(name)}".encode("utf-8")).hexdigest()[:6].upper()
    print("=" * 46)
    print(f"🎉 恭喜 {name}！{_LEVEL} 通關！")
    print(f"🔑 通關密語：PYQ-{_LEVEL}-{code}")
    print("👉 回到入口網頁，把密語貼到這一關的「輸入通關密語」欄位。")
    print("=" * 46)

print(f"🧰 魔法工具箱已準備好！本關有 {len(_TASKS)} 個任務：{'、'.join(_TASKS)}")
print("   做完每個任務後，執行它下方的「檢查」格；全部通過後執行最下方的「通關密語」。")

# ---------------- 各任務的檢查規則 ----------------
def _check_10_1(run):
    out, ns = run()
    import os
    if not os.path.isfile('data/names.txt'): return (False, "找不到 data/names.txt。")
    lines = [l.strip() for l in open('data/names.txt', encoding='utf-8') if l.strip()]
    return (lines == ['小明', '小美', '阿華', '小芳'], f"檔案內容應該是四行名字，現在是 {lines}。")
任務定義("10-1", _check_10_1, 提示="f.write(n + '\\n')，\\n 是換行。")

def _check_10_2(run):
    out, ns = run()
    lines = 行列表(out)
    return (lines == ['1. 小明', '2. 小美', '3. 阿華', '4. 小芳'], f"應該印出 1. 小明 ... 4. 小芳，現在是 {lines}。")
任務定義("10-2", _check_10_2, 提示="print(f'{i}. {line.strip()}')。")

def _check_10_3(run):
    # 先重置檔案，避免重複附加
    with open('data/names.txt', 'w', encoding='utf-8') as f:
        f.write('小明\n小美\n阿華\n小芳\n')
    out, ns = run()
    if "'a'" not in run.src and '"a"' not in run.src: return (False, "附加要用 'a' 模式。")
    content = [l.strip() for l in open('data/names.txt', encoding='utf-8') if l.strip()]
    return (content[-1] == '大雄' and len(content) == 5 and 出現(out, "共5人"), f"檔案最後應該是大雄、共 5 人，現在是 {content}。")
任務定義("10-3", _check_10_3, 提示="open(..., 'a', ...)；len(lines) 是行數。")

def _check_10_4(run):
    out, ns = run("2.4", "24", "0", "24", "5")
    if not 出現(out, "請輸入整數"): return (False, "輸入 2.4 應該印出 輸入錯誤, 請輸入整數。")
    if not 出現(out, "除數為0"): return (False, "除數 0 應該印出 除數為0, 請重新輸入。")
    return (出現(out, "24/5=4.80"), "最後應該印出 24 / 5 = 4.80。")
任務定義("10-4", _check_10_4, 提示="兩個 except 分別印出對應訊息。")

def _check_10_5(run):
    out, ns = run()
    lines = 行列表(out)
    if not any(ln.startswith('滑鼠') and ln.endswith('150.00') for ln in lines): return (False, "滑鼠那一行應該以 150.00 結尾。")
    return (ns.get("total") == 2530 and 出現(out, "總金額2,530.00"), "總金額應該是 2,530.00。")
任務定義("10-5", _check_10_5, 提示="total += qty * price。")


## 📁 10-1　檔案在哪裡？Colab 的檔案系統
- Colab 的工作目錄是 `/content`，點左邊的 📁 圖示可以看到檔案；**重新連線後檔案會消失**（要保留請下載或存到雲端硬碟）。
- 課本的範例用 `c:/data/`（Windows 路徑）；在 Colab 改用相對路徑（`data/stu.txt`）即可。
- `os` 模組負責路徑與資料夾：`os.path.exists()`、`os.path.isfile()`、`os.mkdir()`、`os.listdir()`。

In [ ]:
import os
print(os.getcwd())                       # 目前工作目錄
if not os.path.exists('data'):
    os.mkdir('data')                     # 建立資料夾（課本 9.2）
print(os.path.isdir('data'), os.listdir('.'))

## 10-2　寫檔與讀檔（課本 9.3–9.4）
```python
f = open('檔名', 模式, encoding='utf-8')   # 模式：'w' 寫入(覆蓋) / 'a' 附加 / 'r' 讀取
f.write('...')  /  f.read()  /  f.readline()  /  f.readlines()
f.close()                                   # 一定要關！
```
更好的寫法是 **`with`**：離開區塊會自動關檔，不怕忘記。

In [ ]:
with open('data/stu.txt', 'w', encoding='utf-8') as fw:      # 課本 ex09/write01.py
    fw.write('王一心, 85, 90\n')
    fw.write('張三飛, 75, 87\n')
    fw.write('周五瑞, 92, 71')
with open('data/stu.txt', 'a', encoding='utf-8') as fa:      # 附加
    fa.write('\n趙七海, 66, 87')
with open('data/stu.txt', 'r', encoding='utf-8') as fr:      # 讀取
    print(fr.read())
print('----- 一行一行讀 -----')
with open('data/stu.txt', encoding='utf-8') as fr:
    for line in fr:                                           # 課本 ex09/read04.py
        name, s1, s2 = line.strip().split(',')
        print(f'{name} 平均 {(int(s1) + int(s2)) / 2:.1f}')

## 10-3　例外處理：程式出錯也不要當掉（課本 9.5）
```python
try:
    可能出錯的程式
except 錯誤類型 as e:
    出錯時要做的事
else:
    沒出錯才做的事（可省略）
finally:
    不管有沒有錯都會做（可省略）
```
常見例外：`ZeroDivisionError`（除以 0）、`ValueError`（轉型失敗，如 `int('abc')`）、`FileNotFoundError`、`IndexError`、`KeyError`、`TypeError`。

In [ ]:
def div(n1, n2):                                  # 課本 ex09/try02.py
    try:
        res = n1 / n2
        print(f'{n1} / {n2} = {res}')
    except Exception as e:
        print('錯誤類型 :', e)
    finally:
        print('執行 finally 敘述')
div(8, 0)
div(8, 5)
arr = [0] * 5                                     # 課本 ex09/try03.py：多個 except
try:
    arr[9] = 90
except ZeroDivisionError:
    print('除數為零')
except IndexError:
    print('串列註標超出範圍')
try:
    open('not_exist.txt')
except FileNotFoundError:
    print('檔案不存在')

### 🎯 任務 10-1　寫入名單

把 `names` 串列寫進 `data/names.txt`，**一行一個名字**（用 `'w'` 模式、`with`、`encoding='utf-8'`）。

In [ ]:
# 🎯 任務 10-1　寫入名單（請保留這一行）
import os
if not os.path.exists('data'):
    os.mkdir('data')
names = ['小明', '小美', '阿華', '小芳']
with open('data/names.txt', 'w', encoding='utf-8') as f:
    for n in names:
        ???
print('寫入完成')

In [ ]:
檢查("10-1")   # ◀ 執行這一格，看看任務 10-1 有沒有過關

### 🎯 任務 10-2　讀出並編號

讀取 `data/names.txt`，印出 `1. 小明`、`2. 小美`……（用迴圈讀每一行，記得 `strip()` 去掉換行）。

In [ ]:
# 🎯 任務 10-2　讀出並編號（請保留這一行）
with open('data/names.txt', 'r', encoding='utf-8') as f:
    i = 1
    for line in f:
        ???
        i += 1

In [ ]:
檢查("10-2")   # ◀ 執行這一格，看看任務 10-2 有沒有過關

### 🎯 任務 10-3　附加一筆資料

用 `'a'` 模式把 `大雄` 加到 `data/names.txt` 最後，然後重新讀取，印出總共幾個名字：`共 5 人`。

In [ ]:
# 🎯 任務 10-3　附加一筆資料（請保留這一行）
with open('data/names.txt', ???, encoding='utf-8') as f:
    f.write('大雄\n')
with open('data/names.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()
print(f"共 {???} 人")

In [ ]:
檢查("10-3")   # ◀ 執行這一格，看看任務 10-3 有沒有過關

### 🎯 任務 10-4　安全除法

讀取被除數與除數（都應該是整數）：輸入不是整數印 `輸入錯誤, 請輸入整數`；除數是 0 印 `除數為0, 請重新輸入`；用 `while` 重複直到成功，最後印 `24 / 5 = 4.80`。

In [ ]:
# 🎯 任務 10-4　安全除法（請保留這一行）
while True:
    try:
        a = int(input('輸入被除數：'))
        b = int(input('輸入除數：'))
        result = a / b
        break
    except ValueError:
        print(???)
    except ZeroDivisionError:
        print(???)
print(f"{a} / {b} = {result:.2f}")

In [ ]:
檢查("10-4")   # ◀ 執行這一格，看看任務 10-4 有沒有過關

### 🎯 任務 10-5　商品報表

`data/products.txt` 每行是 `品名,數量,單價`。讀取後印出格式化報表（品名寬 8 靠左、數量寬 6、單價寬 8 小數 2 位），最後印 `總金額 2,530.00`。

In [ ]:
# 🎯 任務 10-5　商品報表（請保留這一行）
with open('data/products.txt', 'w', encoding='utf-8') as f:
    f.write('滑鼠,10,150\n鍵盤,5,30\n螢幕,2,440\n')
total = 0
with open('data/products.txt', 'r', encoding='utf-8') as f:
    for line in f:
        name, qty, price = line.strip().split(',')
        qty, price = int(qty), float(price)
        total += ???
        print(f"{name:<8}{qty:>6}{price:>8.2f}")
print(f"總金額 {total:,.2f}")

In [ ]:
檢查("10-5")   # ◀ 執行這一格，看看任務 10-5 有沒有過關

## 💡 挑戰題（不計分）
用 `import json` 把一個字典 `json.dump()` 到檔案，再 `json.load()` 讀回來。JSON 是網路資料交換最常見的格式。

---
## 🔑 通關密語

全部任務都 ✅ 之後，執行下面這一格，會得到你專屬的通關密語（和暱稱綁定，每個人不一樣）。

In [ ]:
通關密語()

---
### 🧭 接下來
**下一關：📈 L11 matplotlib 繪製圖表** → [在 Colab 開啟](https://colab.research.google.com/github/johnnychao/python-quest-2026/blob/main/notebooks/L11_matplotlib_charts.ipynb)

回到入口網頁：https://johnnychao.github.io/python-quest-2026/